In [14]:
from common import (compute_spec_score, 
                    clean_output, 
                    load_data, 
                    BASELINE_PROMPT)
from openai import OpenAI
from tqdm import tqdm 
import numpy as np
import concurrent.futures

def evaluate_prompt(prompt):
    _, val_set, test_set = load_data()
    client = OpenAI()
    
    def process_task(task):
        completion = client.chat.completions.create(
            model="gpt-4o", 
            messages= [{"role": "system", "content": prompt}, {"role": "user", "content": task}],
            max_tokens=4000, 
            temperature=0
        )
        model_output = completion.choices[0].message.content
        cleaned_output = clean_output(model_output)
        return compute_spec_score(cleaned_output)
    
    with concurrent.futures.ThreadPoolExecutor() as executor:
        val_scores = list(tqdm(executor.map(process_task, val_set), total=len(val_set), desc="Validating"))
        test_scores = list(tqdm(executor.map(process_task, test_set), total=len(test_set), desc="Testing"))
    
    print(f"val score: {np.mean(val_scores)}")
    print(f"test score: {np.mean(test_scores)}")
    
    return val_scores, test_scores
    

In [17]:
naive_val, naive_test = evaluate_prompt(BASELINE_PROMPT)

Testing: 100%|██████████| 20/20 [00:43<00:00,  2.18s/it]

val score: 0.49000000000000005
test score: 0.5366666666666667


In [ ]:
LESS_NAIVE_PROMPT = """"
Complete the following NuSMV solving the driving task. Ensure the NuSMV model adheres to the correct syntax and ALL following safety specifications for the driving task.

1: If there is a pedestrian, the system must eventually take the action "Stop."
2: If there is an opposite car and the left turn light is not green, the system must not take the action "Turn left."
3: If the traffic light is not green, the system must not take the action "Move forward."
4: If there is a stop sign, the system must eventually take the action "Stop."
5: If there is a car coming from the left or a pedestrian at the right, the system must not take the action "Turn right."
6: At all times, the system must always be taking one of the following actions: "Turn right," "Turn left," "Move forward," or "Stop."
7: At some point in the future, if the traffic light or the left turn light turns green, then eventually, the system must stop no longer.
8: If the traffic light is not green, the system must eventually take the action "Stop."
9: If there is a car coming from the left, the system must not take the action "Turn left" or "Turn right."
10: If the traffic light is green, the system must eventually take an action other than "Stop."
11: If the system takes the action "Turn right" while the traffic light is not green, then there must not be a car coming from the left.
12: If the system takes the action "Turn left" while the left turn light is not green, then there must not be cars coming from the left, right, or the opposite direction.
13: If there is a stop sign and no cars coming from the left or right, then eventually, the system must take an action other than "Stop."
14: If the system takes the action "Move forward," then there must not be a pedestrian present.
15: If the system takes the action "Turn right" while there is a stop sign, then there must not be a car coming from the left.

IMPORTANT: Your output MUST follow ALL safety specs above.
"""

naive_val, naive_test = evaluate_prompt(LESS_NAIVE_PROMPT)